# Data Ingestion with LangChain

Loading documents from different sources, then splitting them into chunks
ready for embedding.

1. `TextLoader`     - plain `.txt` files
2. `PyPDFLoader`    - PDFs, one `Document` per page
3. `WebBaseLoader`  - HTML scraped from a URL
4. Text splitters   - turn long documents into chunks

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

# LangSmith tracing (optional - only used if the keys are set in .env)
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

print("env loaded")

env loaded


## 1. TextLoader

Reads a single text file and returns a list with one `Document` in it.

In [2]:
# text loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("speech.txt", encoding="utf-8")
loader

C:\Users\ap036\AppData\Local\Temp\ipykernel_26048\1021814718.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
text_documents = loader.load()
text_documents

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the\ntested foundations of political liberty. We have no selfish ends to serve. We\ndesire no conquest, no dominion. We seek no indemnities for ourselves, no\nmaterial compensation for the sacrifices we shall freely make.\n\nWe are but one of the champions of the rights of mankind. We shall be satisfied\nwhen those rights have been made as secure as the faith and the freedom of\nnations can make them.\n\nIt is a fearful thing to lead this great peaceful people into war, into the most\nterrible and disastrous of all wars, civilization itself seeming to be in the\nbalance. But the right is more precious than peace, and we shall fight for the\nthings which we have always carried nearest our hearts.\n\nTo such a task we can dedicate our lives and our fortunes, everything that we\nare and everything that we have, with the pride of those who know that the day\n

Every loader returns a list of `Document` objects. A `Document` has two
attributes worth knowing:

- `page_content` - the text itself
- `metadata`     - where the text came from

In [4]:
doc = text_documents[0]

print("number of documents :", len(text_documents))
print("metadata            :", doc.metadata)
print("characters          :", len(doc.page_content))
print()
print(doc.page_content[:300])

number of documents : 1
metadata            : {'source': 'speech.txt'}
characters          : 1124

The world must be made safe for democracy. Its peace must be planted upon the
tested foundations of political liberty. We have no selfish ends to serve. We
desire no conquest, no dominion. We seek no indemnities for ourselves, no
material compensation for the sacrifices we shall freely make.

We are


## 2. PyPDFLoader

`PyPDFLoader` returns **one `Document` per page**, and records the page
number in the metadata.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader("attention.pdf")
pdf_documents = pdf_loader.load()

print("pages loaded:", len(pdf_documents))
pdf_documents[0]
pdf_documents

pages loaded: 2


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'attention.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Attention Is All You Need - Abstract\nThe dominant sequence transduction models are based on complex\nrecurrent or convolutional neural networks that include an encoder\nand a decoder. The best performing models also connect the encoder\nand decoder through an attention mechanism.\nWe propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence\nand convolutions entirely.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'attention.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='Page 2 - Results\nExperiments on two machine translation tasks show these models to\nbe superior in quality while being more parallelizable and requiring\nsignificantly less time to train.\nOur model achieves 28.4 B

In [6]:
for page in pdf_documents:
    print(f"--- page {page.metadata['page']} ({len(page.page_content)} chars) ---")
    print(page.page_content[:200].strip())
    print()

--- page 0 (432 chars) ---
Attention Is All You Need - Abstract
The dominant sequence transduction models are based on complex
recurrent or convolutional neural networks that include an encoder
and a decoder. The best performin

--- page 1 (345 chars) ---
Page 2 - Results
Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring
significantly less time to train.
Our model achi



## 3. WebBaseLoader

Scrapes a web page. Passing a `bs4.SoupStrainer` keeps only the parts of the
HTML you care about, instead of the whole page including nav bars and footers.

In [7]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

web_loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-title", "post-content", "post-header")
        )
    ),
)

web_documents = web_loader.load()

print("documents :", len(web_documents))
print("characters:", len(web_documents[0].page_content))
print("source    :", web_documents[0].metadata["source"])
print()
print(web_documents[0].page_content[:300].strip())

USER_AGENT environment variable not set, consider setting it to identify your requests.


documents : 1
characters: 43047
source    : https://lilianweng.github.io/posts/2023-06-23-agent/

LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as i


In [8]:
from langchain_community.document_loaders import ArxivLoader

In [9]:
docs = ArxivLoader(
    query="1706.03762",  # arxiv id or a search query
    load_max_docs=1,
).load()

print("docs:", len(docs))
print(docs[0].metadata)


docs: 1
{'Published': '2023-08-02', 'Title': 'Attention Is All You Need', 'Authors': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin', 'Summary': 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our m

## 4. Splitting documents into chunks

Loaded documents are usually too long to embed in one piece.
`RecursiveCharacterTextSplitter` is the general-purpose choice: it tries to
split on paragraphs first, then lines, then words, so chunks break at natural
boundaries wherever possible.

`chunk_overlap` repeats a little text between neighbouring chunks so a sentence
that straddles a boundary is not lost.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(text_documents)

print("1 document ->", len(chunks), "chunks")
chunks[0]

1 document -> 3 chunks


Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the\ntested foundations of political liberty. We have no selfish ends to serve. We\ndesire no conquest, no dominion. We seek no indemnities for ourselves, no\nmaterial compensation for the sacrifices we shall freely make.\n\nWe are but one of the champions of the rights of mankind. We shall be satisfied\nwhen those rights have been made as secure as the faith and the freedom of\nnations can make them.')

In [11]:
for i, chunk in enumerate(chunks):
    print(f"--- chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content)
    print()

--- chunk 0 (471 chars) ---
The world must be made safe for democracy. Its peace must be planted upon the
tested foundations of political liberty. We have no selfish ends to serve. We
desire no conquest, no dominion. We seek no indemnities for ourselves, no
material compensation for the sacrifices we shall freely make.

We are but one of the champions of the rights of mankind. We shall be satisfied
when those rights have been made as secure as the faith and the freedom of
nations can make them.

--- chunk 1 (293 chars) ---
It is a fearful thing to lead this great peaceful people into war, into the most
terrible and disastrous of all wars, civilization itself seeming to be in the
balance. But the right is more precious than peace, and we shall fight for the
things which we have always carried nearest our hearts.

--- chunk 2 (355 chars) ---
To such a task we can dedicate our lives and our fortunes, everything that we
are and everything that we have, with the pride of those who know that

The same splitter works on any loader's output, so the PDF and the web page go
through exactly the same call.

In [12]:
pdf_chunks = splitter.split_documents(pdf_documents)
web_chunks = splitter.split_documents(web_documents)

print(f"speech.txt   : {len(text_documents):>3} docs -> {len(chunks):>3} chunks")
print(f"attention.pdf: {len(pdf_documents):>3} docs -> {len(pdf_chunks):>3} chunks")
print(f"web page     : {len(web_documents):>3} docs -> {len(web_chunks):>3} chunks")

speech.txt   :   1 docs ->   3 chunks
attention.pdf:   2 docs ->   2 chunks
web page     :   1 docs -> 134 chunks


### Splitting raw strings

If you have text rather than `Document` objects, use `split_text`, which
returns plain strings.

In [13]:
raw_text = text_documents[0].page_content

string_chunks = splitter.split_text(raw_text)

print(type(string_chunks[0]).__name__, "-", len(string_chunks), "chunks")
print(string_chunks[0][:200])

str - 3 chunks
The world must be made safe for democracy. Its peace must be planted upon the
tested foundations of political liberty. We have no selfish ends to serve. We
desire no conquest, no dominion. We seek no 


## Summary

| Source | Loader | Documents returned |
|---|---|---|
| `.txt` | `TextLoader` | 1 per file |
| `.pdf` | `PyPDFLoader` | 1 per page |
| URL | `WebBaseLoader` | 1 per URL |

All of them return `Document` objects, so the same
`RecursiveCharacterTextSplitter` handles every source. The chunks are what you
would hand to an embedding model and store in a vector database next.